# Kohei Identical Twins Bias Detection Notebook

This notebook trains a propensity score matching engine and evaluates fairness metrics
on the synthetic Kohei loan dataset. It is designed to run top-to-bottom in Google Colab.


## Section 0: Setup
Install required packages, import libraries, and mount Google Drive for exports.


In [ ]:
!pip -q install scikit-learn pandas numpy shap dice-ml aif360 matplotlib seaborn onnx skl2onnx xgboost


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KDTree
from sklearn.metrics import confusion_matrix
from sklearn.feature_selection import mutual_info_classif

import shap
import xgboost as xgb

from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.datasets import BinaryLabelDataset

import dice_ml

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

sns.set_theme(style="whitegrid")
np.random.seed(42)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Section 1: Load Data
Load the synthetic dataset, inspect distributions, and visualize approval bias by group.


In [ ]:
# Update this path if you uploaded the file elsewhere in Drive.
DATA_PATH = Path('/content/drive/MyDrive/kohei/full_loan_data.csv')

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').transpose().head(12)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.histplot(df['annual_income'], bins=30, ax=axes[0, 0])
axes[0, 0].set_title('Annual Income (INR)')
sns.histplot(df['cibil_score'], bins=30, ax=axes[0, 1])
axes[0, 1].set_title('CIBIL Score')
sns.histplot(df['loan_amount'], bins=30, ax=axes[0, 2])
axes[0, 2].set_title('Loan Amount (INR)')
sns.histplot(df['foir_ratio'], bins=30, ax=axes[1, 0])
axes[1, 0].set_title('FOIR Ratio')
sns.histplot(df['loan_to_value'], bins=30, ax=axes[1, 1])
axes[1, 1].set_title('Loan-to-Value')
sns.histplot(df['age'], bins=30, ax=axes[1, 2])
axes[1, 2].set_title('Age')
plt.tight_layout()


In [ ]:
def plot_approval_by_group(df, col):
    rates = df.groupby(col)['approved'].mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=rates.index, y=rates.values)
    plt.title(f'Approval Rate by {col}')
    plt.ylabel('Approval Rate')
    plt.xticks(rotation=30)
    plt.ylim(0, 1)
    plt.show()

for col in ['social_category', 'religion', 'gender']:
    plot_approval_by_group(df, col)

df['age_group'] = pd.cut(df['age'], bins=[20, 35, 50, 60, 75], labels=['21-35', '36-50', '51-60', '60+'])
plot_approval_by_group(df, 'age_group')


## Section 2: Identical Twins Matching Engine
The PropensityScoreMatcher learns creditworthiness from financial features only, then pairs
applicants across protected groups with similar propensity scores.


In [ ]:
class PropensityScoreMatcher:
    def __init__(self, caliper=0.1):
        # Matching threshold for propensity score distance.
        self.caliper = caliper
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, X_financial, y_financial):
        # Train logistic regression on financial features only.
        self.model.fit(X_financial, y_financial)
        scores = self.model.predict_proba(X_financial)[:, 1]
        return scores

    def match_pairs(self, df, protected_col, propensity_scores):
        # Use KDTree for efficient nearest-neighbor matching.
        df = df.copy()
        df['propensity_score'] = propensity_scores

        # Majority group is the most frequent category.
        majority_group = df[protected_col].value_counts().idxmax()
        minority_df = df[df[protected_col] != majority_group]
        majority_df = df[df[protected_col] == majority_group]

        tree = KDTree(majority_df[['propensity_score']].values)
        distances, indices = tree.query(minority_df[['propensity_score']].values, k=1)

        matches = []
        for i, (distance, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
            if distance <= self.caliper:
                minority_row = minority_df.iloc[i]
                majority_row = majority_df.iloc[idx]
                matches.append({
                    'protected_col': protected_col,
                    'minority_group': minority_row[protected_col],
                    'majority_group': majority_row[protected_col],
                    'minority_index': minority_row.name,
                    'majority_index': majority_row.name,
                    'minority_score': minority_row['propensity_score'],
                    'majority_score': majority_row['propensity_score'],
                    'score_distance': distance
                })
        return pd.DataFrame(matches)

    def compute_divergence(self, df, matched_pairs, decision_col):
        # Calculate approval rate divergence between matched twins.
        minority_outcomes = df.loc[matched_pairs['minority_index'], decision_col].values
        majority_outcomes = df.loc[matched_pairs['majority_index'], decision_col].values

        minority_rate = minority_outcomes.mean() if len(minority_outcomes) else 0
        majority_rate = majority_outcomes.mean() if len(majority_outcomes) else 0
        air = minority_rate / majority_rate if majority_rate > 0 else 0

        return {
            'minority_rate': minority_rate,
            'majority_rate': majority_rate,
            'air': air,
            'pair_count': len(matched_pairs)
        }

    def evaluate_match_quality(self, before_df, after_df):
        # Compute standardized mean differences across financial features.
        smd = {}
        for col in before_df.columns:
            mean_before = before_df[col].mean()
            mean_after = after_df[col].mean()
            std_pooled = np.sqrt((before_df[col].var() + after_df[col].var()) / 2)
            smd[col] = (mean_after - mean_before) / std_pooled if std_pooled > 0 else 0
        return pd.Series(smd).abs().sort_values(ascending=False)


## Section 3: Run Twin Matching Analysis
Train propensity model, find twin pairs for protected attributes, and visualize divergences.


In [ ]:
financial_cols = [
    'annual_income', 'cibil_score', 'foir_ratio', 'employment_years',
    'loan_amount', 'loan_to_value'
]
protected_cols = ['social_category', 'religion', 'gender', 'age_group']

X_financial = df[financial_cols]
y = df['approved']

matcher = PropensityScoreMatcher(caliper=0.05)
propensity_scores = matcher.fit(X_financial, y)

results = {}
matched_data = {}

for col in protected_cols:
    pairs = matcher.match_pairs(df, col, propensity_scores)
    stats = matcher.compute_divergence(df, pairs, 'approved')
    results[col] = stats
    matched_data[col] = pairs

pd.DataFrame(results).T


In [ ]:
# Visualize a sample of matched twins with different outcomes.
sample_col = 'social_category'
pairs = matched_data[sample_col]

if len(pairs) > 0:
    sample_pairs = pairs.sample(min(10, len(pairs)), random_state=42)
    twin_df = pd.DataFrame({
        'minority_id': df.loc[sample_pairs['minority_index'], 'application_id'].values,
        'minority_group': df.loc[sample_pairs['minority_index'], sample_col].values,
        'minority_decision': df.loc[sample_pairs['minority_index'], 'approved'].values,
        'majority_id': df.loc[sample_pairs['majority_index'], 'application_id'].values,
        'majority_group': df.loc[sample_pairs['majority_index'], sample_col].values,
        'majority_decision': df.loc[sample_pairs['majority_index'], 'approved'].values,
        'score_distance': sample_pairs['score_distance'].values
    })
    twin_df


## Section 4: Standard Bias Metrics (AIF360)
Compute AIR, Equalized Odds, and Statistical Parity Difference.


In [ ]:
def build_aif360_dataset(df, protected_col, privileged_value):
    return BinaryLabelDataset(
        df=df[[protected_col, 'approved']],
        label_names=['approved'],
        protected_attribute_names=[protected_col],
        favorable_label=1,
        unfavorable_label=0
    )

def compute_bias_metrics(df, protected_col, privileged_value):
    dataset = build_aif360_dataset(df, protected_col, privileged_value)
    metric = BinaryLabelDatasetMetric(
        dataset,
        privileged_groups=[{protected_col: privileged_value}],
        unprivileged_groups=[{protected_col: v} for v in df[protected_col].unique() if v != privileged_value]
    )
    return {
        'statistical_parity_difference': metric.statistical_parity_difference(),
        'disparate_impact': metric.disparate_impact()
    }

bias_results = {}
bias_results['gender'] = compute_bias_metrics(df, 'gender', 'Male')
bias_results['religion'] = compute_bias_metrics(df, 'religion', 'Hindu')
bias_results['social_category'] = compute_bias_metrics(df, 'social_category', 'General')

pd.DataFrame(bias_results).T


## Section 5: SHAP Feature Importance
Train XGBoost on the full dataset and inspect feature contributions.


In [ ]:
feature_cols = financial_cols + ['social_category', 'religion', 'gender', 'age', 'pin_code']
target_col = 'approved'

X = df[feature_cols]
y = df[target_col]

categorical_cols = ['social_category', 'religion', 'gender', 'pin_code']
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    tree_method='hist'
)

clf = Pipeline(steps=[('preprocess', preprocess), ('model', model)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict_proba(X_test)[:, 1]
print('AUC:', roc_auc_score(y_test, preds))


In [ ]:
# SHAP analysis
X_sample = X_test.sample(200, random_state=42)

X_transformed = preprocess.fit_transform(X_train)
feature_names = preprocess.get_feature_names_out()

model.fit(X_transformed, y_train)

X_sample_transformed = preprocess.transform(X_sample)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample_transformed)

shap.summary_plot(shap_values, X_sample_transformed, feature_names=feature_names)


## Section 6: Proxy Variable Detection
Compute mutual information between financial features and protected attributes.


In [ ]:
proxy_results = {}
for attr in ['social_category', 'religion', 'gender']:
    y_attr = df[attr].astype('category').cat.codes
    mi = mutual_info_classif(df[financial_cols + ['pin_code']].assign(pin_code=df['pin_code'].astype('category').cat.codes), y_attr)
    proxy_results[attr] = pd.Series(mi, index=financial_cols + ['pin_code'])

proxy_df = pd.DataFrame(proxy_results)
proxy_df


In [ ]:
# Flag features with MI > 0.15
proxy_flags = (proxy_df > 0.15)
proxy_flags


## Section 7: DiCE Counterfactual Generation
Generate counterfactuals for denied minority applicants.


In [ ]:
dice_data = dice_ml.Data(
    dataframe=df[feature_cols + [target_col]],
    continuous_features=financial_cols + ['age'],
    outcome_name=target_col
)

dice_model = dice_ml.Model(model=clf, backend='sklearn')
dice = dice_ml.Dice(dice_data, dice_model, method='random')

denied_minority = df[(df['approved'] == 0) & (df['social_category'].isin(['SC', 'ST']))]
sample_denied = denied_minority.sample(10, random_state=42)

cf_examples = dice.generate_counterfactuals(
    sample_denied[feature_cols],
    total_CFs=1,
    desired_class='opposite'
)

cf_examples.visualize_as_dataframe(show_only_changes=True)


## Section 8: Model Export
Export the propensity model to ONNX and save metadata to Google Drive.


In [ ]:
# Train propensity model on financial features only for ONNX export.
propensity_model = LogisticRegression(max_iter=1000)
propensity_model.fit(X_financial, y)

initial_type = [('float_input', FloatTensorType([None, len(financial_cols)]))]
onnx_model = convert_sklearn(propensity_model, initial_types=initial_type)

export_dir = Path('/content/drive/MyDrive/kohei_exports')
export_dir.mkdir(parents=True, exist_ok=True)

with open(export_dir / 'model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

metadata = {
    'model': 'propensity_logistic_regression',
    'financial_features': financial_cols,
    'caliper': matcher.caliper
}

with open(export_dir / 'feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Exported to:', export_dir)
print('Download via Drive or use firebase storage upload after syncing.')
